In [1]:
import os
import glob
import numpy as np
import pandas as pd
import pickle as pkl
import matplotlib.pyplot as plt
import seaborn as sns
import psychofit as pfit
import ot
from scipy.stats import wasserstein_distance
from manifold.decoding.functions.neurometric import fit_get_shift_range

In [2]:
def get_target_df_engagement(
    target,
    pred,
    trials_df,
    engagement_values=None,
    engagement_col="engagement",
    fit_type="prob_left",
):
    if engagement_values is not None:
        eng_vals = np.asarray(engagement_values).flatten()
        eng_vals = eng_vals[trials_df["mask"]]
    elif trials_df is not None and engagement_col in trials_df.columns:
        eng_vals = trials_df[engagement_col][trials_df["mask"]].values
    else:
        raise ValueError("Must provide 'engagement_values' or 'trials_df' with 'engagement_col'.")

    median_val = np.nanmedian(eng_vals)
    engagement_bin = eng_vals > median_val

    offset = 1 - target.max()
    corr_test = target + offset
    corr_pred = pred + offset
    pred_signs = np.sign(corr_pred)

    df = pd.DataFrame(
        {
            "stimuli": corr_test,
            "predictions": corr_pred,
            "sign": pred_signs,
            "raw_pred": pred,
            "engagement_bin": engagement_bin,
        }
    )

    grpby = df.groupby(["engagement_bin", "stimuli"])
    if fit_type == "prob_left":
        grpbyagg = grpby.agg(
            {
                "sign": [
                    ("num_trials", "count"),
                    ("prop_L", lambda x: ((x == 1).sum() + (x == 0).sum() / 2.0) / len(x)),
                ]
            }
        )
    elif fit_type == "continuous":
        grpbyagg = grpby.agg(
            {
                "raw_pred": [
                    ("num_trials", "count"),
                    ("scaled_mean", lambda x: np.mean((x + 1) / 2.0)),
                ]
            }
        )
    else:
        raise ValueError("Unknown fit_type")

    return [
        grpbyagg.loc[k].reset_index().values.T
        for k in sorted(grpbyagg.index.get_level_values("engagement_bin").unique())
    ]

In [16]:
real_predictions = np.load(
    "../data/ephys_neurometric/CSHL045/034e726f-b35f-41e0-8d6c-a22cc32391fb/LGd_predictions_real.npy"
).squeeze()
pseudo_predictions = np.load(
    "../data/ephys_neurometric/CSHL045/034e726f-b35f-41e0-8d6c-a22cc32391fb/LGd_predictions_pseudo.npy"
).squeeze()

In [17]:
target_real = np.load(
    "../data/ephys_neurometric/CSHL045/034e726f-b35f-41e0-8d6c-a22cc32391fb/targets_real.npy"
).squeeze()
target_pseudo = np.load(
    "../data/ephys_neurometric/CSHL045/034e726f-b35f-41e0-8d6c-a22cc32391fb/targets_pseudo.npy"
).squeeze()